# TP – Recherche de motifs avec les Matchers de spaCy

**Module : Traitement automatique du langage naturel (NLP)**

## Objectifs pédagogiques
À la fin de ce TP, vous serez capable de :
- distinguer `Matcher` et `PhraseMatcher` ;
- construire des motifs à partir des attributs des tokens ;
- utiliser les opérateurs de répétition, notamment `OP: '*'` ;
- rechercher des expressions complètes dans un texte ;
- interpréter `match_id`, `start` et `end` ;
- extraire le contexte autour d'une correspondance ;
- appliquer des règles de recherche à un texte arabe et identifier les limites de la tokenisation.

**Consignes :** complétez les cellules `TODO`, exécutez le notebook dans l'ordre et répondez aux questions d'interprétation.

## 1. Installation et imports

Exécutez les commandes d'installation uniquement si spaCy ou le modèle anglais ne sont pas déjà installés.

In [ ]:
# Décommentez si nécessaire
# %pip install -U spacy
# !python -m spacy download en_core_web_sm

import spacy
from spacy.matcher import Matcher, PhraseMatcher

# TODO 1 : charger le modèle anglais
nlp = ...

## 2. Comprendre les deux types de Matchers

- `Matcher` recherche des motifs définis à partir des propriétés des tokens (forme, minuscules, ponctuation, catégorie grammaticale, etc.).
- `PhraseMatcher` recherche des expressions complètes connues à l'avance, à partir d'une liste de termes.

**Question 1 :** dans quel cas choisiriez-vous `Matcher` plutôt que `PhraseMatcher` ?

## 3. `Matcher` : rechercher des motifs de tokens

Un motif est une liste de dictionnaires. Chaque dictionnaire décrit un token.

- `LOWER` permet de comparer la forme en minuscules.
- `IS_PUNCT` vérifie si le token est un signe de ponctuation.

### 3.1 Définir les motifs

On souhaite détecter les différentes formes de l'expression *solar power* :
- `solarpower` ;
- `solar power` ;
- `solar-power`.

In [ ]:
# TODO 2 : créer un Matcher à partir du vocabulaire de nlp
matcher = ...

# TODO 3 : définir les trois motifs demandés
patterns = [
    ...,
    ...,
    ...
]

# TODO 4 : associer les motifs à la règle "SOLAR_POWER"
...

### 3.2 Appliquer le Matcher

Utilisez le texte ci-dessous. Exécutez le matcher puis affichez, pour chaque correspondance :
- le nom de la règle ;
- les indices `start` et `end` ;
- le texte correspondant.

In [ ]:
text = (
    "The Solar Power industry is growing. "
    "Demand for solarpower is increasing. "
    "Solar-power cars are gaining popularity."
)

# TODO 5 : créer le Doc spaCy
doc = ...

# TODO 6 : exécuter le matcher
matches = ...

# TODO 7 : parcourir les correspondances et afficher règle, indices et texte
for match_id, start, end in ...:
    span = ...
    print(...)

**Question 2 :** pourquoi `end` est-il un indice exclusif ? Que représente `doc[start:end]` ?

### 3.3 Autoriser une ponctuation facultative

L'opérateur `OP: '*'` signifie « zéro ou plusieurs occurrences ».

Modifiez le motif pour reconnaître `solar power`, `solar-power` et des formes contenant éventuellement plusieurs tokens de ponctuation entre `solar` et `power`.

**Travail demandé :**
1. Retirez l'ancienne règle `SOLAR_POWER`.
2. Définissez les nouveaux motifs.
3. Ajoutez-les au matcher.
4. Relancez la recherche sur le même document.
5. Comparez les résultats avec ceux de la section précédente.

In [ ]:
# TODO 8 : retirer l'ancienne règle
...

flexible_patterns = [
    ...,
    ...,
    ...,
    ...
]

# TODO 9 : ajouter les motifs flexibles
...

# TODO 10 : afficher les textes correspondant à la nouvelle règle
for match_id, start, end in ...:
    print(...)

**Question 3 :** quelle différence existe-t-il entre `OP: '*'` et un motif sans opérateur ? Que signifie « zéro ou plusieurs » dans cet exemple ?

## 4. `PhraseMatcher` : rechercher des expressions complètes

Utilisez `PhraseMatcher` lorsque vous connaissez déjà les expressions à rechercher.

### 4.1 Construire une liste d'expressions

Les expressions à rechercher sont :
- `voodoo economics`
- `supply-side economics`
- `trickle-down economics`
- `free-market economics`

**Travail demandé :**
1. Créez la liste des expressions.
2. Initialisez un `PhraseMatcher` avec `attr='LOWER'`.
3. Transformez chaque expression en objet `Doc` avec `nlp.make_doc()`.
4. Ajoutez les motifs au matcher.

In [ ]:
# TODO 11 : définir la liste des expressions
phrase_list = [...]

# TODO 12 : créer le PhraseMatcher (recherche insensible à la casse)
phrase_matcher = ...

# TODO 13 : convertir les expressions en objets Doc
phrase_patterns = [...]

# TODO 14 : ajouter les motifs sous le nom "ECONOMIC_TERM"
...

### 4.2 Rechercher les expressions dans un texte

In [ ]:
sample_text = (
    "Some commentators use the phrase voodoo economics. "
    "Others discuss supply-side economics and free-market economics."
)

# TODO 15 : analyser le texte
phrase_doc = ...

# TODO 16 : rechercher et afficher le nom de la règle et l'expression détectée
for match_id, start, end in ...:
    print(...)

**Question 4 :** à quoi sert `attr='LOWER'` ? Que se passerait-il si on utilisait `attr='ORTH'` ?

### 4.3 Extraire le contexte autour d'une correspondance

On souhaite afficher trois tokens avant et trois tokens après chaque expression détectée.

- `left` correspond à l'indice de début du contexte.
- `right` correspond à l'indice de fin du contexte.
- `max(0, ...)` évite un indice négatif.
- `min(len(phrase_doc), ...)` évite de dépasser la fin du document.

**Travail demandé :** complétez le code afin d'afficher le match, son contexte et une ligne de séparation.

In [ ]:
for match_id, start, end in phrase_matcher(phrase_doc):
    # TODO 17 : calculer l'indice de début du contexte
    left = ...

    # TODO 18 : calculer l'indice de fin du contexte
    right = ...

    # TODO 19 : afficher l'expression et son contexte
    print("Match:", ...)
    print("Context:", ...)
    print("-" * 40)

**Question 5 :** pourquoi faut-il utiliser `max()` et `min()` pour calculer les limites du contexte ?

## 5. Appliquer les Matchers à un texte arabe

Dans cette partie, nous utilisons `spacy.blank("ar")`. Ce pipeline fournit une tokenisation arabe, mais ne contient pas de modèle entraîné pour l'étiquetage morphosyntaxique ou la reconnaissance d'entités.

La recherche repose sur les formes des tokens. L'orthographe et la tokenisation sont donc importantes.

### 5.1 Utiliser `Matcher` pour des variantes de « الذكاء الاصطناعي »

On souhaite détecter plusieurs écritures de l'expression :
- `الذكاءالإصطناعي`
- `الذكاء الإصطناعي`
- `الذكاء الصناعي`

**Travail demandé :**
1. Créez le pipeline arabe minimal.
2. Initialisez un `Matcher`.
3. Définissez les motifs correspondant aux variantes.
4. Analysez le texte arabe et affichez les expressions détectées.

In [ ]:
# TODO 20 : créer le pipeline arabe minimal
nlp_ar = ...

# TODO 21 : créer un Matcher pour l'arabe
arabic_matcher = ...

# TODO 22 : définir les motifs des variantes
arabic_patterns = [
    ...,
    ...,
    ...
]

# TODO 23 : ajouter les motifs
...

arabic_text = (
    "يتميز الذكاءالإصطناعي بأنه يتطور بسرعة. "
    "بدأ الطلاب في دراسة الذكاء الإصطناعي والذكاء الصناعي."
)

# TODO 24 : analyser le texte arabe
arabic_doc = ...

# TODO 25 : afficher les correspondances
for match_id, start, end in ...:
    print(...)

**Question 6 :** pourquoi une variante orthographique peut-elle ne pas être détectée ? Quel rôle joue la tokenisation ?

### 5.2 Utiliser `PhraseMatcher` pour rechercher des noms

On souhaite rechercher les expressions suivantes :
- `بني هاشم`
- `بنو هاشم`
- `قريش`
- `بني المطلب`
- `بنو المطلب`
- `عبد مناف`

**Travail demandé :**
1. Créez la liste des expressions.
2. Initialisez un `PhraseMatcher` avec `attr='ORTH'`.
3. Convertissez les expressions en objets `Doc`.
4. Ajoutez les motifs sous le nom `HISTORICAL_NAMES`.
5. Recherchez les expressions dans le texte fourni.

In [ ]:
arabic_phrases = [...]

# TODO 26 : créer le PhraseMatcher
arabic_phrase_matcher = ...

# TODO 27 : convertir les expressions et ajouter les motifs
arabic_phrase_patterns = [...]
...

arabic_sample = nlp_ar(
    "من الأسماء الواردة: بني هاشم، بنو هاشم، قريش، وبنو المطلب."
)

# TODO 28 : afficher les expressions reconnues
for match_id, start, end in ...:
    print(...)

## 6. Exercices d'application

### Exercice 1 – Énergie renouvelable
Ajoutez des motifs pour reconnaître :
- `wind power`
- `wind-power`

Testez-les sur une phrase contenant les deux formes.

### Exercice 2 – Expressions liées au Machine Learning
Recherchez :
- `machine learning`
- `Machine Learning`
- `machine-learning`

Comparez les résultats avec `attr='LOWER'` et `attr='ORTH'`.

### Exercice 3 – Vocabulaire spécialisé
Choisissez trois expressions techniques de votre domaine (informatique, médecine, économie, biologie, etc.). Construisez un `PhraseMatcher` et recherchez-les dans un paragraphe.

### Exercice 4 – Extraction du contexte
Pour chaque correspondance de l'exercice 3, affichez :
- le texte détecté ;
- les indices des tokens ;
- les trois tokens précédents et les trois tokens suivants, lorsque disponibles.

### Exercice 5 – Variantes en arabe
Ajoutez deux variantes orthographiques d'une expression arabe de votre choix. Testez-les et expliquez les éventuelles différences de tokenisation.

In [ ]:
# TODO 29 : réaliser ici l'exercice 1 ou 2
# Définir les motifs, créer le matcher et tester sur un texte de votre choix.

...

In [ ]:
# TODO 30 : réaliser ici l'exercice 3
# Choisir trois expressions de votre domaine et les rechercher dans un paragraphe.

...

In [ ]:
# TODO 31 : réaliser ici l'exercice 4
# Afficher chaque match avec ses indices et son contexte.

...

## 7. Questions de synthèse

Répondez brièvement :

1. Quelle est la différence entre `Matcher` et `PhraseMatcher` ?
2. Dans quels cas l'attribut `LOWER` est-il utile ?
3. Que signifie `OP: '*'` ?
4. À quoi correspondent `match_id`, `start` et `end` ?
5. Pourquoi extraire le contexte autour d'une expression ?
6. Pourquoi faut-il vérifier la tokenisation avant de construire des motifs pour l'arabe ?
7. Quelle est la limite de `spacy.blank("ar")` dans ce TP ?

## 8. À retenir

- `Matcher` convient aux motifs flexibles au niveau des tokens.
- `PhraseMatcher` est adapté à la recherche d'un vocabulaire d'expressions connues.
- Une correspondance est représentée par `(match_id, start, end)`.
- `LOWER` permet une recherche insensible à la casse.
- `OP: '*'` autorise zéro ou plusieurs occurrences.
- La tokenisation détermine les correspondances possibles.
- Un pipeline arabe vierge permet la tokenisation, mais ne fournit pas d'annotations linguistiques entraînées.